[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheBestProEver/100micron-sim/blob/main/notebooks/00_start_here.ipynb)

# 00 · Start here: what DOLL-E measures, and how its simulator is built

**The question.** What is DOLL-E trying to measure and why, why does it need a rocket, and how is the simulator
that predicts the experiment put together, so that you can read it piece by piece instead of as a black box?

**After this notebook you can explain**
* the one number DOLL-E measures (the speed below which two colliding dust grains stick) and why it matters;
* why the measurement needs the microgravity of a sounding-rocket flight;
* how the full flight simulator (`dolle_sim`) is built from a few pieces: materials, a contact law,
  grains and walls, the flight's loads, the cameras;
* what the small readable core (`dolle_core`) is, how it relates to the full simulator, and how we know the two agree.

**Three ways to run everything.** (1) In the browser: the "Open in Colab" badge at the top of each notebook runs it on
Google Colab straight from GitHub (the first code cell installs the repository there). (2) Locally in Jupyter: from a
clone of the repository, open the `notebooks/` folder. (3) As plain Python: the scripts in `examples/` print the key
numbers, e.g. `python examples/sticking_threshold.py`.

In [1]:
# Make the repository's packages (dolle_core, dolle_sim) importable, in Colab or from a local clone.
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:   # in Colab: install the repository from GitHub (takes ~30 s)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/TheBestProEver/100micron-sim.git"], check=True)
else:                               # locally: the repository folder is this folder or the one above (notebooks/)
    sys.path.insert(0, str(next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "dolle_core").is_dir())))

In [2]:
import math
import dolle_core as dc                       # the readable core: the physics of a collision and of a wall contact
from dolle_core import nb_flight as nbf       # small helpers for these notebooks (tables, comparisons)
from dolle_core import nb_collisions as nb    # helpers for the collision notebooks (here: measured momentum change)

## 1. The question DOLL-E answers

**DOLL-E** (*Dust Observations in Low-gravity and Low-pressure Environments*) is a student sounding-rocket experiment.
Small glass tubules hold dust grains about 100 µm across. In the few minutes of microgravity above ~100 km the grains
are set free, drift, and collide slowly with each other, and cameras record every impact: **did the two grains stick or
bounce, and how fast were they moving towards each other?** From many impacts DOLL-E wants the **sticking speed** (or
sticking threshold): the impact speed below which grains stick. It wants it for **carbon and silicate** grains, two
astrophysically relevant kinds of dust, and how the two differ (README). How to set the grains free slowly enough
is still an open problem: notebooks 04 and 05 show that no flight load frees them and that the release tried so far
(5,000 g kicks) throws them out far faster than the speeds to be measured.

**Why it matters.** Dust in space grows by colliding and sticking ("coagulation"); whether a collision builds a larger
grain or ends in a bounce is decided by exactly this speed. The theory DOLL-E's simulator uses is the one developed
for this problem by Chokshi, Tielens & Hollenbach (1993, "Dust coagulation", *ApJ* 407, 806) and Dominik & Tielens
(1997, "The physics of dust coagulation and the structure of dust aggregates in space", *ApJ* 480, 647): two sticky
elastic spheres (the JKR contact of notebook 01) lose energy every time they touch and separate, and slow grains do
not have enough energy to separate at all. Those papers tabulate the material constants of graphite and quartz
(silica) that the simulator still uses (`dolle_core/materials.py`).

With those constants the theory predicts (two equal grains of radius 50 µm, i.e. 100 µm across; the full derivation is
in notebooks 01 and 03):

In [3]:
v_si = dc.sticking_threshold(dc.make_pair("silica", 50e-6, "silica", 50e-6))        # [m/s]
v_gr = dc.sticking_threshold(dc.make_pair("graphite", 50e-6, "graphite", 50e-6))
print(f"silica   grains stick below {v_si * 1e3:.2f} mm/s")
print(f"graphite grains stick below {v_gr * 1e3:.2f} mm/s  ({v_gr / v_si:.1f} x silica)")
measured = 1.2   # m/s: Poppe, Blum & Henning (2000, ApJ 533, 454), as quoted in the README ("How realistic is it?")
for target in ("silica", "glass_borosilicate"):       # the experiment shot 1.2 um spheres at a flat target
    v_small = dc.sticking_threshold(dc.make_pair("silica", 0.6e-6, target))               # r2 omitted: a flat
    print(f"1.2 um silica sphere on a flat {target:18s}: model {v_small:.3f} m/s, measured ~{measured} m/s "
          f"({measured / v_small:.1f} x)")

silica   grains stick below 6.18 mm/s
graphite grains stick below 38.36 mm/s  (6.2 x silica)
1.2 um silica sphere on a flat silica            : model 0.276 m/s, measured ~1.2 m/s (4.3 x)
1.2 um silica sphere on a flat glass_borosilicate: model 0.265 m/s, measured ~1.2 m/s (4.5 x)


**Why this happens, and why it must be measured.** Graphite comes out about six times stickier than silica: part of
that comes from published material values (surface energy and stiffness) and part from an **assumed** damping input
(the "dry restitution" $e_{\rm dry}$; notebook 03 takes the ratio apart). The theory is only as good as these inputs,
and it is known to miss: for 1.2 µm silica spheres shot at a flat target, where it gives about 0.27 m/s, the
laboratory measurement of Poppe, Blum & Henning (2000; described as spheres "colliding with a target" by Beckwith,
Henning & Nakagawa 2000) found 4.3–4.6 times more (above). The surface energy of dry silica in vacuum is itself
debated: Kimura et al. (2015, *ApJ* 812, 67) argue it is several times the commonly used value, while Kreuzig et al.
(2024, *A&A* 688) found no such rise for water-repellent silica. **Measuring these speeds directly is what DOLL-E is
for.** The carbon DOLL-E will fly is not decided yet; the simulator's carbon is **graphite**, and notebook 03 shows
how the sticking speed would move for other carbons.

## 2. Why it needs microgravity

A sticking speed of a few mm/s means a grain needs about a second to cross a few millimetres of a camera's view. In
that time a constant acceleration $a$ moves it by $\tfrac12 a t^2$ (constant-acceleration kinematics):

In [4]:
t = 1.0                                              # [s], about the time a 6 mm/s grain needs for 6 mm
for name, a_g in (("on the ground, 1 g", 1.0), ("in DOLL-E's microgravity window, 1e-4 g", 1e-4)):
    print(f"{name:42s}: falls {0.5 * a_g * dc.G0 * t**2 * 1e3:8.2f} mm in {t:.0f} s, "
          f"reaching {a_g * dc.G0 * t * 1e3:8.2f} mm/s")

on the ground, 1 g                        : falls  4903.32 mm in 1 s, reaching  9806.65 mm/s
in DOLL-E's microgravity window, 1e-4 g   : falls     0.49 mm in 1 s, reaching     0.98 mm/s


**Why this happens.** On the ground a grain falls metres while it covers millimetres sideways, and it gains speed far
faster than the mm/s that matter. In the microgravity window of a sounding rocket the residual acceleration (about
$10^{-4}$ g, an assumed value, notebook 05) moves it by half a millimetre in the same second: slow collisions can
happen and be filmed. The flight provides a few minutes of this (notebook 05).

## 3. The notebooks, in reading order

| notebook | the question it answers |
|---|---|
| **00 · Start here** (this one) | What does DOLL-E measure, and how is the simulator built? |
| [01 · Two grains colliding head-on](01_two_grains_colliding.ipynb) ([Colab](https://colab.research.google.com/github/TheBestProEver/100micron-sim/blob/main/notebooks/01_two_grains_colliding.ipynb)) | What happens during the microsecond two grains are in contact, and why do slow grains stick and fast ones bounce? The JKR contact law, damping and plastic yield, derived step by step. |
| [02 · Momentum conservation](02_momentum_conservation.ipynb) ([Colab](https://colab.research.google.com/github/TheBestProEver/100micron-sim/blob/main/notebooks/02_momentum_conservation.ipynb)) | Why the total momentum comes out exactly as it went in (Newton's third law), where the code guarantees it, and where the kinetic energy goes. |
| [03 · Silica or carbon](03_silica_vs_carbon.ipynb) ([Colab](https://colab.research.google.com/github/TheBestProEver/100micron-sim/blob/main/notebooks/03_silica_vs_carbon.ipynb)) | What changes when the grains are carbon instead of silica, which material values cause it, and how the sticking speed would move for other carbons. |
| [04 · A grain on the wall](04_grain_on_a_wall.ipynb) ([Colab](https://colab.research.google.com/github/TheBestProEver/100micron-sim/blob/main/notebooks/04_grain_on_a_wall.ipynb)) | What holds a grain on the tubule wall, what it takes to pull it off, slide, roll or twist it, and which flight loads do so. |
| [05 · The flight in brief](05_the_flight_in_brief.ipynb) ([Colab](https://colab.research.google.com/github/TheBestProEver/100micron-sim/blob/main/notebooks/05_the_flight_in_brief.ipynb)) | The flight from lift-off to splashdown as loads on the dust, what the simulator has found so far, and what DOLL-E can measure in the microgravity window. |

## 4. How the full flight simulator is built, on one page

The full simulator `dolle_sim` follows the dust of several tubules through the whole flight on one clock. It is
built from the same few pieces the notebooks explain, plus the machinery that makes a whole flight affordable when
each contact must be followed in nanosecond steps:

```
 MATERIALS            CONTACT LAW              GRAINS AND WALLS IN A TUBULE        FLIGHT LOADS         MEASUREMENT
 materials.py   --->  contact.py (1-D,   --->  engine.py  (v0.6, "full contact"):  flight.py, loads.py  analysis.py
 rho, E, nu,          head-on: JKR +           every grain a sphere with 6          thrust, drag, spin,  synthetic cameras,
 gamma, e_dry, p_y    damping + yield)         degrees of freedom; each grain is    de-spin, re-entry,   speed noise, the
                      dem.py (3-D: + the       in one regime: free flight, full     parachute, splash    sticking-curve fit
                      sliding, rolling and     contact, rolling, slaved cluster     (+ environment.py:
                      twisting springs and     or asleep; every step verified       residual g, gas
                      their limits)            simulation.py (v0.5, event-driven)   drag, charging ...)
                                                          |
                                     mission.py: all tubules, lift-off to splashdown, one clock
```

| piece | full simulator (`dolle_sim/`) | readable core (`dolle_core/`) | notebook |
|---|---|---|---|
| material values | `materials.py` | `materials.py` (the same 16 materials, value for value) | 01, 03 |
| two bodies as one effective contact | `materials.py: pair_props` | `pair.py` | 01 |
| JKR adhesion, damping, plastic yield | `contact.py` (1-D), `dem.py` (3-D) | `jkr.py`, `damping.py`, `plastic.py` | 01 |
| a head-on collision, stick or bounce | `contact.py: JKRContact.resolve, threshold` | `collide.py` | 01, 02, 03 |
| the sticking speed by hand | `contact.py`, `provenance.py` (sensitivities) | `scaling.py` | 03 |
| momentum along a line of grains | `dem.py`, `verify_full.py` (conservation checks) | `many.py` | 02 |
| a grain resting on a wall | `dem.py: make_params, contact_force` | `wall.py` | 04 |
| the flight's loads | `flight.py`, `loads.py` | used directly from `dolle_sim.flight` | 04, 05 |
| many grains in 3-D, regimes, verified steps | `engine.py`, `statics.py`, `slaved.py`, `roll1.py` | not in the core | 00 (below) |
| gas drag, charging, light, the cameras | `environment.py`, `electrostatics.py`, `analysis.py` | not in the core | 05 |

**How the engine decides** what to do with each grain (which regime it is in, when it sleeps, wakes, rolls or joins
the full contact model, and how each step is verified) is written out decision by decision, each with its test, its
reason and the check that verifies it, in [docs/ENGINE_LOGIC.md](../docs/ENGINE_LOGIC.md)
([on GitHub](https://github.com/TheBestProEver/100micron-sim/blob/main/docs/ENGINE_LOGIC.md)). Where every model and number comes from is in
[PROVENANCE.md](../PROVENANCE.md) ([on GitHub](https://github.com/TheBestProEver/100micron-sim/blob/main/PROVENANCE.md)).

The size of each part:

In [5]:
for name, (files, lines) in nbf.package_lines().items():
    print(f"{name:40s} {files:3d} files {lines:7,d} lines")

dolle_sim                                 29 files  15,489 lines
dolle_core (physics)                      10 files   1,296 lines
dolle_core (notebook helpers nb_*.py)      2 files   1,013 lines


**Why this matters.** Most of the full simulator is the machinery for many grains over a whole flight (regimes,
verified steps, checkpoints, outputs); the physics of one contact fits in the few files of `dolle_core`.

## 5. How `dolle_core` relates to `dolle_sim`

`dolle_core` is a **copy of the physics, not a simplification of it**: the same equations, the same material values,
and, for the collision, the same arithmetic in the same order as `dolle_sim/contact.py`. So it gives the same answers,
which `tests/test_dolle_core.py` checks case by case: every value of every material, the pair properties, the sticking
speeds (identical), the restitutions at ten speeds per case including dented (plastic) impacts, the simulator's own
contact checks, momentum and energy bookkeeping, and the wall limits against `dolle_sim/dem.py`
(`python -m pytest tests/test_dolle_core.py`). The same comparison, live:

In [6]:
for g1, g2, wall in (("silica", "silica", False), ("graphite", "graphite", False), ("graphite", "glass_borosilicate", True)):
    c = nbf.core_vs_sim(g1, g2, wall=wall)      # dolle_core vs dolle_sim/contact.py JKRContact, radius 50 um
    print(f"{c['case']:34s} sticking speed: core {c['v_stick_core'] * 1e3:.6f}, simulator {c['v_stick_sim'] * 1e3:.6f} mm/s;"
          f" restitution at 2x: difference {abs(c['e_core'] - c['e_sim']):.1e}")

silica-silica                      sticking speed: core 6.175177, simulator 6.175177 mm/s; restitution at 2x: difference 2.2e-16
graphite-graphite                  sticking speed: core 38.358561, simulator 38.358561 mm/s; restitution at 2x: difference 0.0e+00
graphite-glass_borosilicate(wall)  sticking speed: core 18.233257, simulator 18.233257 mm/s; restitution at 2x: difference 0.0e+00


**Why this happens.** Both codes take the same steps with the same numbers, so they agree to the last digit or to
rounding error (~10⁻¹⁶). This is what lets you read `dolle_core` and know that you are reading what the big code does.

**What the core leaves out, and why the narrower model is enough for what it is used for.** The core treats a
collision as **head-on and one-dimensional**. The full simulator's 3-D contact model (`dolle_sim/dem.py`) adds what
happens in an oblique impact: the sliding, rolling and twisting springs act during the contact and the grains spin.
For **head-on** impacts the 3-D model gives the same sticking speed; the full simulator checks this in
`verify_full.check_dem_vs_1d` (about 15 s; it needs numba, and the first run compiles the 3-D engine, about a
minute). Its 1-D column is `contact.py` at 2000 steps per contact and bisection tolerance 1e-5, so silica shows 6.177
rather than the 6.175 above (tolerance 2e-3: that difference is the bisection's tolerance, not physics):

In [7]:
import importlib.util
if importlib.util.find_spec("numba"):
    from dolle_sim import verify_full
    check = verify_full.check_dem_vs_1d()                 # 3-D contact model vs the 1-D model, head-on
    print(check["name"], "->", "PASS" if check["passed"] else "FAIL")
    print(check["value"], "| criterion:", check["criterion"])
else:
    print("numba is not installed: skipping (pip install numba)")

3-D contact dynamics vs the 1-D contact model -> PASS
silica 6.186 vs 6.177 mm/s; graphite 38.344 vs 38.299 mm/s; worst 1.6e-03 | criterion: < 2e-3


**Why this happens.** In a head-on impact the sliding, rolling and twisting springs are not loaded, so the 3-D model
reduces to the 1-D one; the small difference shrinks when the 3-D step is halved (notebook 01, section 12), so it
is step error. That OBLIQUE impacts stick at the same normal speed is an assumption of the simulator's v0.5 chamber
model (PROVENANCE.md); notebook 01 tests it with the 3-D model at two impact offsets. What else the core does **not**
cover, and where it lives: many grains moving in 3-D with walls, regimes and verified steps (`engine.py`; notebook 04
uses its wall-contact limits, and notebook 04 also runs its 3-D model for a grain rolling on a wall); gas drag,
charging, light pressure and the other forces on free grains (`environment.py`, `electrostatics.py`); the flight's
loads on moving grains (`loads.py`, `engine.py`); the cameras (`analysis.py`). For the sticking speed itself none of
these enter: it is a property of the contact.

## 6. One collision, to see the core at work

`dc.collide` runs one head-on collision step by step (notebook 01 explains every line of it). Two silica grains of
radius 50 µm, one at rest, the other arriving at 0.8 and at 2 times the sticking speed:

In [8]:
for factor in (0.8, 2.0):
    r = dc.collide("silica", 50e-6, "silica", 50e-6, u1=factor * v_si)
    print(f"{factor:3.1f} x sticking speed ({factor * v_si * 1e3:5.2f} mm/s): {r.outcome:6s} restitution {r.e:.3f}, "
          f"in contact {r.contact_time * 1e6:4.2f} us; velocities after {r.v1 * 1e3:6.3f}, {r.v2 * 1e3:6.3f} mm/s; "
          f"largest momentum change at any step (measured) {nb.momentum_drift(r):.0e}")

0.8 x sticking speed ( 4.94 mm/s): stick  restitution 0.000, in contact 0.66 us; velocities after  2.470,  2.470 mm/s; largest momentum change at any step (measured) 1e-15
2.0 x sticking speed (12.35 mm/s): bounce restitution 0.671, in contact 0.89 us; velocities after  2.029, 10.321 mm/s; largest momentum change at any step (measured) 1e-15


**Why this happens.** Below the sticking speed the contact takes more energy than the grains brought, so they stay
together and move on at the centre-of-mass velocity (half the incoming speed, for equal grains); above it they bounce
with a restitution (rebound speed over impact speed) below one. For the stuck pair the velocities printed are SET to
the centre-of-mass velocity once the code has proven the pair bound; the momentum change is measured at every step
of the integration, and either way it is rounding error (notebook 02). "In contact" is, for the stuck pair, the time until the code can prove they will never
separate (notebook 01).

## What to remember

* DOLL-E measures the **sticking speed** of ~100 µm grains, carbon vs silicate; the model predicts a few mm/s for silica
  and a few tens of mm/s for graphite, but its inputs are uncertain and theory has missed measurements by several times.
* How to release the grains slowly enough in flight is still open (notebooks 04 and 05).
* It needs **microgravity**: at 1 g a grain falls metres in the second it needs to cover millimetres.
* The full simulator is **materials → contact law → grains and walls → flight loads → cameras**, with an engine that
  makes the whole flight affordable; [docs/ENGINE_LOGIC.md](../docs/ENGINE_LOGIC.md) explains its decisions.
* `dolle_core` is the same physics in a few readable files, checked against `dolle_sim` by `tests/test_dolle_core.py`;
  the head-on 1-D collision it uses gives the 3-D model's head-on sticking speed.

## Check it yourself

1. Compute the sticking speed of two graphite grains of radius 25 µm and of 50 µm. By what factor does it change
   when the radius doubles? (The hand formula of notebook 03 says $v \propto R^{-5/6}$: compare with $2^{-5/6}$.)
2. Run `python examples/sticking_threshold.py` in a terminal (from the repository folder) and find the same numbers.

In [9]:
# your turn, e.g.:
# v25 = dc.sticking_threshold(dc.make_pair("graphite", 25e-6, "graphite", 25e-6))
# v50 = dc.sticking_threshold(dc.make_pair("graphite", 50e-6, "graphite", 50e-6))
# print(v25, v50, v50 / v25, 2 ** (-5 / 6))